### KNB Data Exploration: Great-tailed Grackle Behavioral Experiments 
---
**Capstone Project · DataTalks.Club 2026**  
**Author:** Victoria Torreno  
**Source:** [KNB Ecoinformatics](https://knb.ecoinformatics.org/view/doi:10.5063/F13B5XBC)  
**Citation:** Logan, C. (2015). Great-tailed grackle behavioral flexibility and problem solving experiments, Santa Barbara, CA USA 2014-2015. Knowledge Network for Biocomplexity.  
**Goal:** Inspect experimental CSV files, understand behavioral metrics, and define the Bronze layer schema.

#### Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path 

In [ ]:
# resolve paths relative to project root 
BASE_DIR = Path().resolve().parents[1] 
WATER_TUBE_PATH = BASE_DIR / 'data' / 'water_tube.csv'
COLOR_TEST_PATH = BASE_DIR / 'data' / 'color_test.csv'
INTERACTION_PATH = BASE_DIR / 'data' / 'interaction.csv'

#### Load Data

**Note:** `water_tube.csv` has no header row. Column names assigned based on the KNB dataset documentation.

In [ ]:
water_tube_columns = [
    'experiment', 'date', 'batch', 'bird', 'trial',
    'choice_number', 'choice_correct', 'choice',
    'extracted_food', 'water_level', 'tube_on_left', 'notes'
]

In [ ]:
water_tube_df = pd.read_csv(WATER_TUBE_PATH, header=None, names=water_tube_columns)
color_test_df = pd.read_csv(COLOR_TEST_PATH)
interaction_df = pd.read_csv(INTERACTION_PATH)

#### Schema & Field Overview

In [ ]:
# preview top 3 records per dataset
water_tube_df.head(3) 

In [ ]:
color_test_df.head(3)

In [ ]:
interaction_df.head(3)

#### Cross-Dataset Bird Consistency

In [ ]:
# cross-dataset bird consistency check
water_birds = set(water_tube_df['bird'].unique())
color_birds = set(color_test_df['Bird'].unique())
interaction_birds = set(interaction_df['Bird'].unique())

In [ ]:
print(f'common across all datasets: \n{water_birds & color_birds & interaction_birds}')
print(f'water tube only: {water_birds - color_birds - interaction_birds}')
print(f'color test only: {color_birds - water_birds - interaction_birds}')
print(f'interaction only: {interaction_birds - color_birds - water_birds}')
print('\nnote: set() means no birds are exclusive to that dataset')

#### Shape & Field Types

In [ ]:
# row and column counts
print(f'water_tube: {water_tube_df.shape}')
print(f'color_test: {color_test_df.shape}')
print(f'interaction: {interaction_df.shape}')

In [ ]:
# field types
print(f'water tube: \n{water_tube_df.dtypes}')
print(f'\ncolor: \n{color_test_df.dtypes}')
print(f'\ninteraction: \n{interaction_df.dtypes}')

#### Descriptive Statistics

In [ ]:
# basic stats
water_tube_df.describe() 

In [ ]:
color_test_df.describe()

In [ ]:
interaction_df.describe()

#### Null Counts

In [ ]:
# null counts
water_tube_df.isnull().sum() 

In [ ]:
color_test_df.isnull().sum()

In [ ]:
interaction_df.isnull().sum()

#### Duplicate Records

In [ ]:
# duplicate records
print(f'water_tube duplicates: {water_tube_df.duplicated().sum()}')
print(f'color_test duplicates: {color_test_df.duplicated().sum()}')
print(f'interaction duplicates: {interaction_df.duplicated().sum()}')

#### Subject & Behavioral Overview

In [ ]:
# unique test subjects per dataset
water_tube_df['bird'].unique() 

In [ ]:
color_test_df['Bird'].unique()

In [ ]:
interaction_df['Bird'].unique()

In [ ]:
# trials per bird
water_tube_df['bird'].value_counts() 

In [ ]:
color_test_df['Bird'].value_counts() 

In [ ]:
interaction_df['Bird'].value_counts()

#### Choice Distribution (watertube)

In [ ]:
# choice distribution (heavy vs light)
water_tube_df['choice'].value_counts() 

#### Correct Choice vs Food Extraction

In [ ]:
# correct choice vs food extraction crosstab
pd.crosstab(water_tube_df['choice_correct'], water_tube_df['extracted_food'])

#### Water Tube Learning Curves

In [ ]:
# plot learning curves
def plot_water_tube(df, bird_col, correct_col, bird_name, trial_col='trial'):
    """plot success rate per trial for a single bird"""
    # set sns theme and define color palette
    sns.set_theme(style='whitegrid', font='sans-serif')
    TEAL = '#8ECAE6'
    SAND = '#FCFAF2'
    SAGE = '#B7B7A4'

    bird_df = df[df[bird_col].str.lower() == bird_name.lower()].copy()
    bird_df = df[df[bird_col] == bird_name].copy()
    bird_df[correct_col] = bird_df[correct_col].map({'Yes': 1, 'No': 0})
    data = bird_df.groupby(trial_col)[correct_col].mean()

    plt.figure(figsize=(9, 4), facecolor='white')
    ax = plt.gca()
    ax.set_facecolor(SAND)
        
    plt.plot(data.index, data.values, 
             color=TEAL, linewidth=3, marker='o',
             markersize=6, markerfacecolor='white', markeredgewidth=2,
             label='success rate')

    plt.fill_between(data.index, data.values, alpha=0.2, color=TEAL)
    plt.axhline(y=0.5, color=SAGE, linestyle=':', linewidth=1.5, label='chance')               
    plt.title(f'Subject Performance: {bird_name}', fontsize=14, fontweight='bold')
    plt.xlabel('trial number', fontsize=10, fontweight='bold')
    plt.ylabel('mean success', fontsize=10, fontweight='bold')
                 
    plt.ylim(-0.05, 1.05)
    plt.xticks(data.index)
    sns.despine(left=True, bottom=True)
    plt.legend(frameon=True, facecolor='white')             
    plt.tight_layout()
    plt.show()

In [ ]:
# learning curves for subjects with the highest data density 
for bird in ['Batido', 'Refresco', 'Cerveza']:
    plot_water_tube(water_tube_df, 'bird', 'choice_correct', bird)

#### Color Test Learning Curves

In [ ]:
def plot_color(df, bird_col, correct_col, bird_name):

    DUSTY_ROSE = '#D4A5A5'
    SAGE = '#9CAF88'
    STONE = '#C9C0BB'
    BACKGROUND = '#F7F4F2'
    TEXT_COLOR = '#4A3F3F'
    LIGHT_TEXT = '#7A6A6A'

    # filter 
    bird_df = df[df[bird_col].str.lower() == bird_name.lower()].copy()
    bird_df[correct_col] = bird_df[correct_col].map({'Yes': 1, 'No': 0})
    bird_df = bird_df.reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(11, 5), facecolor=BACKGROUND, dpi=100)
    ax.set_facecolor(BACKGROUND)

    # jittered scatter
    jitter = np.random.uniform(-0.04, 0.04, size=len(bird_df))
    ax.scatter(bird_df.index, bird_df[correct_col] + jitter,
               alpha=0.35, color=STONE, s=25, edgecolors='none',
               label='trial outcome', zorder=2)

    # rolling average
    rolling_avg = bird_df[correct_col].rolling(window=10, min_periods=1).mean()
    ax.plot(bird_df.index, rolling_avg, color=DUSTY_ROSE, linewidth=3,
            label='10-trial rolling avg', solid_capstyle='round', zorder=3)

    # chance baseline
    ax.axhline(y=0.5, color=SAGE, linestyle='--',
               linewidth=1.5, alpha=0.9, label='chance (50%)', zorder=1)

    # fill above/below chance
    ax.fill_between(bird_df.index, rolling_avg, 0.5,
                    where=(rolling_avg >= 0.5), alpha=0.15, color=DUSTY_ROSE, zorder=1)
    ax.fill_between(bird_df.index, rolling_avg, 0.5,
                    where=(rolling_avg < 0.5), alpha=0.15, color=STONE, zorder=1)

    # title + subtitle
    fig.text(0.12, 1.0, f"{bird_name.capitalize()} — Color Test",
             fontsize=15, fontweight='bold', color=TEXT_COLOR)
    fig.text(0.12, 0.94, 'Rolling success rate across trial sequence',
             fontsize=10, color=LIGHT_TEXT)

    # axes
    ax.set_xlabel('trial sequence', fontsize=11, color=LIGHT_TEXT)
    ax.set_ylabel('success rate', fontsize=11, color=LIGHT_TEXT)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'], color=LIGHT_TEXT)
    ax.tick_params(colors=LIGHT_TEXT)
    ax.set_ylim(-0.1, 1.15)

    # spines
    for spine in ax.spines.values():
        spine.set_edgecolor('#DDD5D0')

    # legend
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1),
              frameon=False, fontsize=10, labelcolor=TEXT_COLOR)

    plt.tight_layout()
    plt.show()

In [ ]:
# compare learning curves for the top 3 subjects
for bird in ['Horchata', 'Refresco', 'Tequila']:
    plot_color(color_test_df, 'Bird', 'CorrectChoice', bird)

#### Interaction Spatial Preference

In [ ]:
def plot_interaction(df, bird_col, interaction_col, birds):
    PEACH = '#FFDAB9'
    SAGE = '#C8D5B9'
    BUTTER = '#FFF9E6'
    TEXT_COLOR = '#5C4A3A'
    LIGHT_TEXT = '#8B7355'
    colors = [PEACH, SAGE]

    fig, axes = plt.subplots(1, len(birds), figsize=(14, 7), facecolor=BUTTER)

    for ax, bird_name in zip(axes, birds):
        ax.set_facecolor(BUTTER)

        # filter
        bird_df = df[df[bird_col].str.lower() == bird_name.lower()].copy()
        approach_counts = bird_df[interaction_col].value_counts()

        wedges, texts, autotexts = ax.pie(
            approach_counts.values,
            labels=None,
            autopct='%1.1f%%',
            startangle=90,
            colors=colors,
            pctdistance=0.75,
            wedgeprops={'edgecolor': BUTTER, 'linewidth': 5, 'width': 0.5}
        )

        # style percentage text
        for autotext in autotexts:
            autotext.set_fontsize(13)
            autotext.set_fontweight('bold')
            autotext.set_color(TEXT_COLOR)

        # donut hole
        center_circle = plt.Circle((0, 0), 0.50, fc=BUTTER)
        ax.add_artist(center_circle)

        # center label
        ax.text(0, 0.08, f'{len(bird_df)}', ha='center', va='center',
                fontsize=26, fontweight='bold', color=TEXT_COLOR)
        ax.text(0, -0.18, 'total trials', ha='center', va='center',
                fontsize=10, color=LIGHT_TEXT)

        # custom legend
        legend_labels = approach_counts.index.tolist()
        handles = [plt.Rectangle((0, 0), 1, 1, fc=c, edgecolor='none') for c in colors]
        ax.legend(handles, legend_labels, loc='lower center',
                  bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False,
                  fontsize=11, labelcolor=TEXT_COLOR)

        # per-chart title
        ax.set_title(f"{bird_name.capitalize()}'s Spatial Preference",
                     fontsize=13, fontweight='bold', color=TEXT_COLOR, pad=15)

    # shared subtitle
    fig.text(0.5, 0.02, 'First Approach Direction',
             ha='center', fontsize=10, color=LIGHT_TEXT)

    plt.tight_layout()
    plt.show()

In [ ]:
# analyze interaction patterns for top-trial subjects
plot_interaction(interaction_df, 'Bird', 'ApproachedFirst', ['Batido', 'Horchata'])

#### Data Exploration Summary 
**Target Species:** Great-tailed Grackle (*Quiscalus mexicanus*)  

---
**Schema**
- `water_tube.csv` has no header row — column names assigned manually based on KNB documentation.
- Column names in `color_test` and `interaction` are not snake_case — will be standardized in Silver layer.
- Mixed types across datasets: dates stored as strings, binary fields (`choice_correct`, `CorrectChoice`) stored as `str` — will require casting during transformation.

**Subjects**
- 6 birds appear across all three datasets: Batido, Cerveza, Horchata, Margarita, Refresco, Tequila.
- 2 birds (`Jugo`, `Michelada`) appear only in `color_test` — likely added in a later experiment batch.
- No birds are exclusive to `water_tube` or `interaction`.

**Volume**
- `water_tube`: 678 rows × 12 columns
- `color_test`: 1,140 rows × 11 columns
- `interaction`: 81 rows × 7 columns — smallest dataset, limited to spatial preference trials.

**Descriptive Stats**
- `water_tube`: trials range from 1–20, water level ranges from 6–13cm.
- `color_test`: overall mean correct rate is 63.4% — above chance (50%), suggesting subjects learned the color association.
- `interaction`: max 10 trials per subject — very small experiment scope.

**Data Quality**
- No duplicate records across all three datasets.
- `water_tube`: `extracted_food`, `water_level`, `tube_on_left`, and `notes` have high null rates — contextual fields, not always recorded. Will drop `notes` in Silver layer.
- `color_test`: `NonoverlappingWindow4-trialbins`, `Criterion`, and `Preference? Notes` are mostly null (862, 1026, 1042 missing respectively) — will drop during transformation.
- `interaction`: only `PutMoreFoodOn` has nulls (64/81) — likely not applicable for all trial types.

**Behavioral Analysis**
- **Water tube (choice distribution):** Subjects strongly preferred Heavy objects (361) over Light (167), consistent with the Aesop's Fable paradigm. `N` (80) and `W` (70) values suggest directional choices in a separate experiment phase within the same file.
- **Correct choice vs food extraction (crosstab):** When choice was correct, food was extracted 126/183 times (69%). When incorrect, food was still extracted 30/121 times (25%) — suggesting occasional accidental successes.
- **Water tube learning curves:** All three subjects (Batido, Refresco, Cerveza) fluctuate around or below chance in early trials, with Refresco showing the clearest upward trend by trial 19–20.
- **Color test learning curves:** All three subjects (Horchata, Refresco, Tequila) show a cyclical pattern — periods of high performance followed by sharp drops — consistent with reversal learning design where the correct color is periodically switched.
- **Spatial preference (interaction):** Both Batido and Horchata show a dominant approach direction (W at ~47% and ~36% respectively), suggesting spatial bias rather than random exploration.

**Trial Counts**
- `water_tube`: Batido (149), Refresco (146), Cerveza (142) have the most data — Tequila (73) and Horchata (63) have notably fewer trials.
- `color_test`: Horchata (210) leads, Batido (80) has the fewest. Jugo and Michelada (130 each) only appear here.
- `interaction`: Evenly distributed across subjects (13–15 trials each).